# 28 · Load: job content (O*NET 31.0), AI exposure (Eloundou et al.) and Career Clusters → `staging`

**Source cards:** `onet.md`, `eloundou_ai_exposure.md`, `career_clusters.md`. Explored in: notebooks 11, 13, 14. Decisions: D5 (O*NET newcomer share), D6 (clusters many-to-many; majors get clusters through their jobs), D7 (GPT-4 β as the AI score).

| Table | One row = | Expected rows |
|---|---|---|
| `stg_onet__occupation` | O*NET job | 1,016 |
| `stg_onet__descriptor_rating` | O*NET job × skill or knowledge element × scale (IM / LV) | 123,760 |
| `stg_onet__education_experience` | O*NET job × question (RL education, RW experience) × answer category | 19,007 |
| `stg_onet__job_zone` | O*NET job | 923 |
| `stg_onet__software` | O*NET job × software product × commodity | 31,821 |
| `stg_eloundou__job_exposure` | O*NET job | 923 |
| `stg_eloundou__task_exposure` | O*NET job × task | 19,265 |
| `stg_cte__cluster` | sub-cluster | 72 |
| `stg_cte__soc_subcluster` | job × sub-cluster | 1,203 |
| `stg_cte__naics_subcluster` | sector × sub-cluster | 112 |
| `stg_cte__cip_subcluster` | major × sub-cluster (reference only, D6) | 4,725 |

All national, one edition each. The roll-ups (O*NET job → SOC job, aggregate codes → parts, pooled AI β) happen in core.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "28-load-skills-ai-clusters.ipynb"
RELEASE = "31.0"
CODE = "O*NET-SOC Code"
conn = connect()
summary = []

of = raw_files("onet")
OZ = of[of.file.str.endswith("db_31_0_csv.zip")].iloc[0]
z = zipfile.ZipFile(OZ.path)
def onet(name: str) -> pd.DataFrame:
    return pd.read_csv(z.open(f"db_31_0_csv/{name}.csv"), dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())

def yn(s: pd.Series, name: str) -> pd.Series:
    assert set(s) <= {"Y", "N", ""}, f"{name}: {set(s)}"
    return s.map({"Y": True, "N": False, "": None}).astype("boolean")

def onet_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, format="%m/%Y", errors="raise")      # O*NET dates are MM/YYYY

def dec(s: pd.Series, name: str) -> pd.Series:
    blank = s == ""
    if blank.any():
        print(f"  {name}: blank -> NULL: {blank.sum():,}")
    return pd.to_numeric(s.mask(blank), errors="raise").astype("float64")

## 1. O*NET occupations (issue 1)

SOC 2018 = the first 7 characters of the O*NET-SOC code (`15-1252.00` → `15-1252`). `has_ratings` = the job has skill ratings (jobs added recently aren't rated yet: shown as "not yet rated by O*NET").

In [2]:
occ = onet("occupation_data")
ess, tra, kno = onet("essential_skills"), onet("transferable_skills"), onet("knowledge")
occ = occ.rename(columns={CODE: "onet_soc_code", "Title": "title", "Description": "description"})
occ["soc_code"] = occ.onet_soc_code.str[:7]
assert occ.onet_soc_code.str.fullmatch(r"\d\d-\d{4}\.\d\d").all()
occ["is_specialty"] = ~occ.onet_soc_code.str.endswith(".00")
occ["has_ratings"] = occ.onet_soc_code.isin(set(ess[CODE]))
occ["onet_release"] = RELEASE
print(f"{len(occ)} O*NET jobs -> {occ.soc_code.nunique()} SOC codes | rated: {occ.has_ratings.sum()} | specialties: {occ.is_specialty.sum()}")
summary.append(stage(occ.assign(source_file=OZ.file), "stg_onet__occupation", ["onet_soc_code"], expected_rows=1_016,
                     source_key="onet", notebook=NB, conn=conn, comment="O*NET 31.0 occupations; soc_code = left(onet_soc_code, 7)."))

1016 O*NET jobs -> 867 SOC codes | rated: 910 | specialties: 149
  stg_onet__occupation: 1,016 rows x 8 cols, key ['onet_soc_code'] unique | local data/interim/staging/stg_onet__occupation.parquet | DB: 1016


## 2. Skills and knowledge ratings (issues 5–7)

Skills come in two tables (10 essential `2.A`, 25 transferable `2.B`), stacked with `domain`. Importance (`IM`, 1–5) and level (`LV`, 0–7) per element. O*NET's own quality flags (`Recommend Suppress`, `Not Relevant`) are kept as booleans; the rating date is kept and shown.

In [3]:
R = pd.concat([ess.assign(domain="skill_essential"), tra.assign(domain="skill_transferable"), kno.assign(domain="knowledge")],
              ignore_index=True)
print(R.groupby(["domain", "Scale ID"]).size().unstack())
assert set(R["Scale ID"]) == {"IM", "LV"}
rat = pd.DataFrame({
    "onet_soc_code": R[CODE], "soc_code": R[CODE].str[:7], "domain": R.domain, "element_id": R["Element ID"],
    "element_name": R["Element Name"], "scale": R["Scale ID"], "value": dec(R["Data Value"], "value"),
    "n": dec(R["N"], "n").astype("Int64"), "std_error": dec(R["Standard Error"], "std_error"),
    "ci_low": dec(R["Lower CI Bound"], "ci_low"), "ci_high": dec(R["Upper CI Bound"], "ci_high"),
    "suppress": yn(R["Recommend Suppress"], "suppress"), "not_relevant": yn(R["Not Relevant"], "not_relevant"),
    "rated_date": onet_date(R["Date"]), "rating_source": R["Domain Source"], "onet_release": RELEASE, "source_file": OZ.file})
print("Suppress = Y by domain:", rat.groupby("domain").suppress.mean().mul(100).round(1).to_dict(), "%")
summary.append(stage(rat, "stg_onet__descriptor_rating", ["onet_soc_code", "element_id", "scale"], expected_rows=123_760,
                     source_key="onet", notebook=NB, conn=conn,
                     comment="O*NET 31.0 skills (essential + transferable) and knowledge, importance (IM 1-5) and level (LV 0-7)."))

Scale ID               IM     LV
domain                          
knowledge           30030  30030
skill_essential      9100   9100
skill_transferable  22750  22750
  n: blank -> NULL: 594
  std_error: blank -> NULL: 16,962
  ci_low: blank -> NULL: 17,698


  ci_high: blank -> NULL: 17,698


Suppress = Y by domain: {'knowledge': 11.3, 'skill_essential': 0.1, 'skill_transferable': 0.3} %


  stg_onet__descriptor_rating: 123,760 rows x 17 cols, key ['onet_soc_code', 'element_id', 'scale'] unique | local data/interim/staging/stg_onet__descriptor_rating.parquet | DB: 123760


## 3. Education and experience shares (D5 newcomer share; issue 9)

`RL` = required level of education (12 categories), `RW` = related work experience (11 categories): the value is the % of workers giving that answer. About 23 respondents per job, so the dashboard shows bands. Category labels come from O*NET's category tables.

In [4]:
edu, tex = onet("education"), onet("training_and_experience")
cats = pd.concat([onet("education_categories"), onet("training_and_experience_categories")])
E = pd.concat([edu[edu["Scale ID"] == "RL"], tex[tex["Scale ID"] == "RW"]], ignore_index=True)
log("education RL + training/experience RW rows", len(edu) + len(tex), len(E))
E = E.merge(cats[["Element ID", "Scale ID", "Category", "Category Description"]], on=["Element ID", "Scale ID", "Category"], how="left")
assert E["Category Description"].notna().all(), "category without a label"
ee = pd.DataFrame({
    "onet_soc_code": E[CODE], "soc_code": E[CODE].str[:7], "question": E["Element Name"], "element_id": E["Element ID"],
    "scale": E["Scale ID"], "category": E.Category.astype(int).astype("Int64"), "category_label": E["Category Description"],
    "share_pct": dec(E["Data Value"], "share_pct"), "n": dec(E["N"], "n").astype("Int64"),
    "suppress": yn(E["Recommend Suppress"], "suppress"), "rated_date": onet_date(E["Date"]),
    "onet_release": RELEASE, "source_file": OZ.file})
sums = ee.groupby(["onet_soc_code", "scale"]).share_pct.sum()
print(f"Shares per job and question sum to {sums.min():.1f}-{sums.max():.1f}%"); assert sums.between(99, 101).all()
summary.append(stage(ee, "stg_onet__education_experience", ["onet_soc_code", "scale", "category"], expected_rows=19_007,
                     source_key="onet", notebook=NB, conn=conn,
                     comment="O*NET 31.0 required education (RL) and related experience (RW): % of workers per answer category."))

  education RL + training/experience RW rows: 38,307 -> 19,007 rows (-19,300)
Shares per job and question sum to 100.0-100.0%


  stg_onet__education_experience: 19,007 rows x 13 cols, key ['onet_soc_code', 'scale', 'category'] unique | local data/interim/staging/stg_onet__education_experience.parquet | DB: 19007


## 4. Job zones and software (issues 4, 6)

In [5]:
jz = onet("job_zones")
jz = pd.DataFrame({"onet_soc_code": jz[CODE], "soc_code": jz[CODE].str[:7], "job_zone": jz["Job Zone"].astype(int).astype("Int64"),
                   "rated_date": onet_date(jz.Date), "rating_source": jz["Domain Source"], "onet_release": RELEASE,
                   "source_file": OZ.file})
assert jz.job_zone.between(1, 5).all()
summary.append(stage(jz, "stg_onet__job_zone", ["onet_soc_code"], expected_rows=923, source_key="onet", notebook=NB, conn=conn,
                     comment="O*NET 31.0 job zone (1 = little preparation ... 5 = extensive)."))

sw = onet("software_skills")
sw = pd.DataFrame({"onet_soc_code": sw[CODE], "soc_code": sw[CODE].str[:7], "product": sw["Workplace Example"],
                   "commodity_id": sw["Element ID"], "commodity_title": sw["Element Name"],
                   "is_hot_technology": yn(sw["Hot Technology"], "hot").astype(bool),
                   "is_in_demand": yn(sw["In Demand"], "in_demand").astype(bool), "onet_release": RELEASE, "source_file": OZ.file})
print(f"{sw["product"].nunique():,} distinct products | hot: {sw.is_hot_technology.mean():.0%} of rows | in demand: {sw.is_in_demand.mean():.0%}")
summary.append(stage(sw, "stg_onet__software", ["onet_soc_code", "product", "commodity_id"], expected_rows=31_821,
                     source_key="onet", notebook=NB, conn=conn, comment="O*NET 31.0 software used on the job, with hot-technology and in-demand flags."))

  stg_onet__job_zone: 923 rows x 7 cols, key ['onet_soc_code'] unique | local data/interim/staging/stg_onet__job_zone.parquet | DB: 923


8,753 distinct products | hot: 36% of rows | in demand: 8%


  stg_onet__software: 31,821 rows x 9 cols, key ['onet_soc_code', 'product', 'commodity_id'] unique | local data/interim/staging/stg_onet__software.parquet | DB: 31821

## 5. AI exposure, Eloundou et al. "GPTs are GPTs" (D7; issues 1–2, 6–7)

Job scores: `dv_rating_beta` (GPT-4 β, the D7 score) and the human-rated β, plus α and γ. Tasks: the leftover index column is dropped, `Task ID` (decimal text) cast to int, blank `Task Type` kept as NULL (weighted 0.5 in core). Tie-out: every published GPT-4 β rebuilds from the task labels with E1 = 1, E2 = 0.5, core tasks weight 1 and supplemental/blank 0.5 (nb 14).

In [6]:
ef = raw_files("eloundou_ai_exposure")
ep = {Path(f).name: (f, p) for f, p in zip(ef.file, ef.path)}
j = pd.read_csv(ep["occ_level.csv"][1], dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
job = pd.DataFrame({"onet_soc_code": j[CODE], "soc_code": j[CODE].str[:7], "title": j.Title,
                    **{f"{k}_gpt4": dec(j[f"dv_rating_{k}"], k) for k in ["alpha", "beta", "gamma"]},
                    **{f"{k}_human": dec(j[f"human_rating_{k}"], k) for k in ["alpha", "beta", "gamma"]},
                    "label_vintage": "GPT-4, 2023 labels", "source_file": ep["occ_level.csv"][0]})
assert job.beta_gpt4.between(0, 1).all()

t = pd.read_csv(ep["full_labelset.tsv"][1], sep="\t", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
assert (t.iloc[:, 0] == t.index.astype(str)).all(), "first column isn't a plain row index"
tid = pd.to_numeric(t["Task ID"], errors="raise")
assert (tid % 1 == 0).all()
print("Task Type:", t["Task Type"].replace("", "(blank)").value_counts().to_dict())
for c in ["gpt4_exposure", "human_exposure_agg"]:
    assert set(t[c]) <= {"E0", "E1", "E2"}, set(t[c])
task = pd.DataFrame({"onet_soc_code": t[CODE], "soc_code": t[CODE].str[:7], "task_id": tid.astype("Int64"), "task": t.Task,
                     "task_type": t["Task Type"].replace("", None), "gpt4_label": t.gpt4_exposure,
                     "human_label": t.human_exposure_agg, "source_file": ep["full_labelset.tsv"][0]})

# Tie-out: rebuild beta_gpt4 from the task labels
w = task.task_type.map({"Core": 1.0}).fillna(0.5)
e = task.gpt4_label.map({"E0": 0.0, "E1": 1.0, "E2": 0.5})
rebuilt = (w * e).groupby(task.onet_soc_code).sum() / w.groupby(task.onet_soc_code).sum()
cmp = job.set_index("onet_soc_code").beta_gpt4.to_frame("published").join(rebuilt.rename("rebuilt"))
cmp["diff"] = (cmp.published - cmp.rebuilt).abs()
print(f"GPT-4 beta rebuilt from tasks: max difference {cmp['diff'].max():.6f} over {len(cmp)} jobs "
      f"(published values are rounded to 2 decimals, so up to 0.005 is rounding)")
display(cmp[cmp["diff"] > 0.0049])
assert (cmp["diff"] <= 0.005 + 1e-9).all()
summary.append(stage(job, "stg_eloundou__job_exposure", ["onet_soc_code"], expected_rows=923, source_key="eloundou_ai_exposure",
                     notebook=NB, conn=conn, comment="Eloundou et al. 2024 AI exposure per O*NET job (GPT-4 and human alpha/beta/gamma). Research estimate."))
summary.append(stage(task, "stg_eloundou__task_exposure", ["onet_soc_code", "task_id"], expected_rows=19_265,
                     source_key="eloundou_ai_exposure", notebook=NB, conn=conn,
                     comment="Eloundou et al. task labels E0/E1/E2 (GPT-4 and human). beta = core 1 / other 0.5 weighted mean of E1=1, E2=0.5."))

Task Type: {'Core': 13225, 'Supplemental': 4728, '(blank)': 1312}
GPT-4 beta rebuilt from tasks: max difference 0.000000 over 923 jobs (published values are rounded to 2 decimals, so up to 0.005 is rounding)


,published,rebuilt,diff
onet_soc_code,,,


  stg_eloundou__job_exposure: 923 rows x 11 cols, key ['onet_soc_code'] unique | local data/interim/staging/stg_eloundou__job_exposure.parquet | DB: 923


  stg_eloundou__task_exposure: 19,265 rows x 8 cols, key ['onet_soc_code', 'task_id'] unique | local data/interim/staging/stg_eloundou__task_exposure.parquet | DB: 19265


## 6. Career Clusters, Advance CTE (D6; issues 1, 6)

The three separate files are used (the combined file has 14 header rows and 2 duplicates). `Sub-Cluster` names are unique across clusters; one NAICS row spells `water systems` in lower case, so every name is matched case-insensitively to the SOC file's spelling. CIP codes keep their leading zeros (read as text).

In [7]:
cf = raw_files("career_clusters")
def cte(name: str, sheet: str):
    row = cf[cf.file.str.endswith(name)].iloc[0]
    return pd.read_excel(row.path, sheet_name=sheet, dtype=str).apply(lambda s: s.str.strip()), row.file

soc, f_soc = cte("SOC-Career-Clusters.xlsx", "SOC - CC - Sub-Clusters")
cip, f_cip = cte("CIP-Career-Clusters.xlsx", "CIP - CC - Sub-Clusters")
nai, f_nai = cte("NAICS-Career-Clusters-Subclusters.xlsx", "CC - Sub-Cluster - NAICS")

canon = {s.lower(): s for s in soc["Sub-Cluster"]}
for name, d in [("CIP", cip), ("NAICS", nai)]:
    fixed = d["Sub-Cluster"] != d["Sub-Cluster"].str.lower().map(canon)
    print(f"{name}: sub-cluster names re-cased to the SOC file's spelling: {sorted(d.loc[fixed, 'Sub-Cluster'].unique())}")
    assert d["Sub-Cluster"].str.lower().isin(canon).all()
    d["Sub-Cluster"] = d["Sub-Cluster"].str.lower().map(canon)

cl = pd.concat([d[["Career Cluster", "Sub-Cluster"]] for d in (soc, cip, nai)]).drop_duplicates()
cl = cl.rename(columns={"Career Cluster": "cluster_name", "Sub-Cluster": "subcluster_name"}).sort_values(["cluster_name", "subcluster_name"])
assert not cl.subcluster_name.duplicated().any(), "a sub-cluster sits in two clusters"
print(f"{cl.cluster_name.nunique()} clusters, {len(cl)} sub-clusters")
summary.append(stage(cl.assign(source_file=f_soc).reset_index(drop=True), "stg_cte__cluster", ["subcluster_name"], expected_rows=72,
                     source_key="career_clusters", notebook=NB, conn=conn, comment="Advance CTE Career Clusters: 14 clusters, 72 sub-clusters."))

for d, f, table, cols, key, n in [
        (soc, f_soc, "stg_cte__soc_subcluster", {"SOC Code": "soc_code", "Occupation Title": "soc_title"}, "soc_code", 1_203),
        (nai, f_nai, "stg_cte__naics_subcluster", {"2-Digit NAICS": "naics_sector", "NAICS title": "naics_title"}, "naics_sector", 112),
        (cip, f_cip, "stg_cte__cip_subcluster", {"CIP Code (2020)": "cip_code", "CIP Title (2020)": "cip_title"}, "cip_code", 4_725)]:
    out = d.rename(columns={**cols, "Career Cluster": "cluster_name", "Sub-Cluster": "subcluster_name"})
    out = out[[*cols.values(), "cluster_name", "subcluster_name"]].assign(source_file=f)
    pattern = {"soc_code": r"\d\d-\d{4}", "cip_code": r"\d\d\.\d{4}", "naics_sector": r"\d\d(-\d\d)?"}[key]
    assert out[key].str.fullmatch(pattern).all(), out[~out[key].str.fullmatch(pattern)][key].unique()
    summary.append(stage(out.reset_index(drop=True), table, [key, "subcluster_name"], expected_rows=n, source_key="career_clusters",
                         notebook=NB, conn=conn, comment=f"Advance CTE crosswalk: {key} x sub-cluster (many-to-many, D6; never sum clusters)."))

CIP: sub-cluster names re-cased to the SOC file's spelling: ['Water Systems']
NAICS: sub-cluster names re-cased to the SOC file's spelling: ['Water Systems']
14 clusters, 72 sub-clusters
  stg_cte__cluster: 72 rows x 3 cols, key ['subcluster_name'] unique | local data/interim/staging/stg_cte__cluster.parquet | DB: 72
  stg_cte__soc_subcluster: 1,203 rows x 5 cols, key ['soc_code', 'subcluster_name'] unique | local data/interim/staging/stg_cte__soc_subcluster.parquet | DB: 1203
  stg_cte__naics_subcluster: 112 rows x 5 cols, key ['naics_sector', 'subcluster_name'] unique | local data/interim/staging/stg_cte__naics_subcluster.parquet | DB: 112
  stg_cte__cip_subcluster: 4,725 rows x 5 cols, key ['cip_code', 'subcluster_name'] unique | local data/interim/staging/stg_cte__cip_subcluster.parquet | DB: 4725


## 7. Summary

In [8]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_onet__occupation,1016,8,onet_soc_code,onet,28-load-skills-ai-clusters.ipynb,1016,2026-10-14T21:22:31+00:00
1,stg_onet__descriptor_rating,123760,17,onet_soc_code + element_id + scale,onet,28-load-skills-ai-clusters.ipynb,123760,2026-10-14T21:22:32+00:00
2,stg_onet__education_experience,19007,13,onet_soc_code + scale + category,onet,28-load-skills-ai-clusters.ipynb,19007,2026-10-14T21:22:33+00:00
3,stg_onet__job_zone,923,7,onet_soc_code,onet,28-load-skills-ai-clusters.ipynb,923,2026-10-14T21:22:33+00:00
4,stg_onet__software,31821,9,onet_soc_code + product + commodity_id,onet,28-load-skills-ai-clusters.ipynb,31821,2026-10-14T21:22:33+00:00
5,stg_eloundou__job_exposure,923,11,onet_soc_code,eloundou_ai_exposure,28-load-skills-ai-clusters.ipynb,923,2026-10-14T21:22:33+00:00
6,stg_eloundou__task_exposure,19265,8,onet_soc_code + task_id,eloundou_ai_exposure,28-load-skills-ai-clusters.ipynb,19265,2026-10-14T21:22:33+00:00
7,stg_cte__cluster,72,3,subcluster_name,career_clusters,28-load-skills-ai-clusters.ipynb,72,2026-10-14T21:22:34+00:00
8,stg_cte__soc_subcluster,1203,5,soc_code + subcluster_name,career_clusters,28-load-skills-ai-clusters.ipynb,1203,2026-10-14T21:22:34+00:00
9,stg_cte__naics_subcluster,112,5,naics_sector + subcluster_name,career_clusters,28-load-skills-ai-clusters.ipynb,112,2026-10-14T21:22:34+00:00
